## 📎 Code this notebook uses
*(how to run, and which code each step uses: [`README.md`](README.md))*

- **`code/pipeline/compute/`** (in this order): `retrieve_kd.py`, `compute_common_epoch.py`,
  `compute_diviner_closure.py`, `compute_review_diagnostics.py`, `compute_albedo_diagnostics.py`,
  `compute_albedo_sensitivity.py`, `compute_albedo_anchor.py`, **`compute_joint_albedo_fit.py`**,
  `compute_joint_valley.py`, `compute_joint_probe_checks.py`, `compute_joint_block_bootstrap.py`
- **`code/pipeline/figures/`:** `make_joint_figures.py` (Figure 4)
- **`code/src/lunar/`:** `equilibrium` (the solver), `properties` (Hayne's $K(T,z)$)

# 03 — The joint fit: albedo and $K_d$ together

**The main result of the paper** (Section 2.6, Table 1, Figure 4). At each
site the model is solved for 25 albedos × 94 values of $K_d$, and both are
fitted to three measurements at once: the sensor temperatures, the measured
mean surface temperature, and the annual-wave diffusivity of Langseth et al.
(1976). The diffusivity is what separates the two: the temperatures alone
trade a brighter surface against better-conducting ground.

In [ ]:
# How much to run (README.md explains the three modes):
#   "show"  : run nothing, read the stored results            (minutes)
#   "quick" : run the short steps and the fast form of long ones  (~30 min for all notebooks)
#   "full"  : run everything from scratch, as for the paper    (about 6 h for all notebooks)
MODE = "show"

import pandas as pd
from lunar import notebook as nb
nb.set_mode(MODE)
SITES_ = ("A15", "A17")

## 1. The analyses the fit builds on

These come from the study's earlier, fixed-albedo stage. The joint fit and
the later checks reuse their outputs: the sensor data and depth handling
(`retrieve_kd.py`), the common-1974 window (`compute_common_epoch.py`), the
Diviner comparison (`compute_diviner_closure.py`, `compute_review_diagnostics.py`)
and the published albedo models (the three `compute_albedo_*` scripts, SI Table S4).
In "quick" mode the long ones keep their stored results.

In [ ]:
DIVINER = ("data/diviner/gcp/global_cumul_avg_cyl_20n30n_002.tab", "data/diviner/gcp/global_cumul_avg_cyl_10n20n_002.tab")
nb.step("compute/retrieve_kd.py", minutes=95, writes="kd_retrieval_results.json")
nb.step("compute/compute_common_epoch.py", minutes=15, writes="common_epoch_sensitivity.json")
nb.step("compute/compute_diviner_closure.py", minutes=3, writes="diviner_closure.json", needs=DIVINER)
nb.step("compute/compute_review_diagnostics.py", minutes=5, writes="review_diagnostics.json", needs=DIVINER)
nb.step("compute/compute_albedo_diagnostics.py", minutes=13, writes="albedo_diagnostics.json")
nb.step("compute/compute_albedo_sensitivity.py", minutes=35, writes="albedo_sensitivity.json")
nb.step("compute/compute_albedo_anchor.py", minutes=5, writes="albedo_anchor.json")

## 2. The joint fit (Section 2.6, Table 1)

Full run about 25 min. "quick" re-analyses the stored grid of model runs
(`joint_albedo_fit_cache.npz`) in seconds: the same minimum, the same
1500-draw bootstrap.

In [ ]:
nb.step("compute/compute_joint_albedo_fit.py", minutes=25, writes="joint_albedo_fit.json",
        quick=("--reuse",), quick_needs="results/joint_albedo_fit_cache.npz")

In [ ]:
jf = nb.load("joint_albedo_fit.json")
rows = []
for s in SITES_:
    for fit in ("with_diffusivity", "temperature_only"):
        b, bt = jf["sites"][s][fit]["best"], jf["sites"][s][fit]["bootstrap"]
        rows.append((s, "temperatures + surface mean + diffusivity" if fit == "with_diffusivity" else "temperatures + surface mean only",
                     round(b["A"], 4), f'{bt["A"]["p2.5"]:.3f} – {bt["A"]["p97.5"]:.3f}',
                     round(b["kd_star_mW"], 2), f'{bt["kd_mW"]["p2.5"]:.2f} – {bt["kd_mW"]["p97.5"]:.2f}',
                     round(b["surface_mean_K"], 1), b["surface_mean_obs_K"]))
display(pd.DataFrame(rows, columns=["site", "fitted to", "albedo A", "A, 95%", "K_d (mW/m/K)", "K_d, 95%",
                                    "model surface mean (K)", "measured (K)"]))
print("Without the diffusivity the K_d intervals are several times wider: that is the albedo trade-off.")

In [ ]:
chk = nb.Checks("Compared with the paper (Table 1, Section 3.2):")
for s, A, kd, lo, hi in (("A15", "0.136", "4.9", "4.3", "5.5"), ("A17", "0.137", "5.9", "5.3", "6.5")):
    w = jf["sites"][s]["with_diffusivity"]
    chk.add(f"{s}: albedo A*", w["best"]["A"], A, ".3f")
    chk.add(f"{s}: K_d*", w["best"]["kd_star_mW"], kd, ".1f")
    chk.add(f"{s}: K_d 95% lower", w["bootstrap"]["kd_mW"]["p2.5"], lo, ".1f")
    chk.add(f"{s}: K_d 95% upper", w["bootstrap"]["kd_mW"]["p97.5"], hi, ".1f")
c = jf["contrast"]["with_diffusivity"]
chk.add("A17 - A15 (bootstrap median)", c["median"], "+1.0", "+.1f")
chk.add("A17 - A15, 95% lower", c["p2_5"], "+0.1", "+.1f")
chk.add("A17 - A15, 95% upper", c["p97_5"], "+1.9", "+.1f")
chk.show()

## 3. What the diffusivity adds: the worked example (SI Table S9)

In [ ]:
nb.step("compute/compute_joint_valley.py", minutes=0.2, writes="joint_valley.json")
v = nb.load("joint_valley.json")
display(pd.DataFrame([(s, round(v[s]["A"], 4), round(v[s]["kd_temperatures_only_mW"], 2),
                       round(v[s]["kd_with_diffusivity_mW"], 2)) for s in SITES_],
                     columns=["site", "albedo (fit node)", "K_d, temperatures only", "K_d, with diffusivity"]))

## 4. Each probe on its own, and the ±5 K Apollo test (Sections 3.3, 4.1)

In [ ]:
nb.step("compute/compute_joint_probe_checks.py", minutes=0.3, writes="joint_probe_checks.json")
pc = nb.load("joint_probe_checks.json")["sites"]
for s in SITES_:
    p = pc[s]
    print(f"Apollo {s[1:]}:")
    print("  K_d from each probe's annual-wave diffusivity:",
          ", ".join(f"{x['kd_mW']:.2f}" for x in p["per_probe_annual"]), "mW m^-1 K^-1")
    print("  K_d from the transient (drilling) diffusivities:",
          " to ".join(f"{x:.2f}" for x in p["transient_kd_range_mW"]))
    k = p["keihm_test_A_fitted"]
    print(f"  K_d values that pass the +-5 K test at the fitted albedo: {k['kd_pass_mW']}")

## 5. Bootstrap with the sensors grouped by probe (SI Table S6)

The production bootstrap resamples the sensors of a site freely. Two grouped
versions: "stratified" resamples them within each probe; "probe_block" does
that and also gives all the sensors of a probe one shared depth offset. The SI
Table S6 caption reports the second, the stricter one.

In [ ]:
nb.step("compute/compute_joint_block_bootstrap.py", minutes=1, writes="joint_block_bootstrap.json")
bb = nb.load("joint_block_bootstrap.json")["schemes"]
rows, shift = [], 0.0
for sc in ("production", "stratified", "probe_block"):
    for s in SITES_:
        k = bb[sc][s]["kd_mW"]
        rows.append((sc, s, round(k["p50"], 2), round(k["p2.5"], 2), round(k["p97.5"], 2)))
        if sc == "probe_block":
            p = bb["production"][s]["kd_mW"]
            shift = max(shift, abs(k["p2.5"] - p["p2.5"]), abs(k["p97.5"] - p["p97.5"]))
display(pd.DataFrame(rows, columns=["scheme", "site", "K_d median", "95% lower", "95% upper"]))
chk = nb.Checks("Compared with the paper (SI Table S6 caption):")
chk.add("probe_block: largest shift of a 95% bound", shift, "0.04", ".2f")
chk.show()

## 6. Figure 4: what fixes the albedo and $K_d$

In [ ]:
nb.figures("figures/make_joint_figures.py", makes="fig_joint_constraints", minutes=2)
nb.show("fig_joint_constraints")

Next: **04_uncertainty_and_tests.ipynb**.